In [ ]:
!pip -q install pdfplumber transformers faiss-cpu openai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

pdf_files_path = '/content/drive/MyDrive/DL/RAG_data'

if not os.path.isdir(pdf_files_path):
    raise FileNotFoundError(
        f"Folder not found: {pdf_files_path}"
    )

pdf_files = sorted([
    f for f in os.listdir(pdf_files_path)
    if f.lower().endswith('.pdf')
])

if not pdf_files:
    raise ValueError("No PDF files found in the folder.")

print("PDF files found:", len(pdf_files))
for file in pdf_files:
    print(file)

PDF files found: 3
case-report-1.pdf
case-report-2.pdf
case-report-3.pdf


Extract text and create document chunks

In [ ]:
import pdfplumber

def extract_pdf_chunks(pdf_path, chunk_words=180, overlap=40):
    chunks = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            words = text.split()

            if not words:
                continue

            step = chunk_words - overlap

            for start in range(0, len(words), step):
                chunk_text = " ".join(
                    words[start:start + chunk_words]
                )

                if chunk_text.strip():
                    chunks.append({
                        "text": chunk_text,
                        "source": os.path.basename(pdf_path),
                        "page": page_number
                    })

    return chunks


documents = []

for filename in pdf_files:
    path = os.path.join(pdf_files_path, filename)
    documents.extend(extract_pdf_chunks(path))

if not documents:
    raise ValueError(
        "No text was extracted. The PDFs may be scanned images "
        "and may require OCR."
    )

print("Total chunks:", len(documents))
print("\nFirst chunk:")
print(documents[0]["text"][:500])
print("Source:", documents[0]["source"])
print("Page:", documents[0]["page"])

Total chunks: 78

First chunk:
CASE STUDY EXAMPLE INTRODUCTION Craniosacral Therapy is a gentle, hands-on form of manual therapy which addresses dysfunction within the craniosacral system, which follows the movement and flow of cerebrospinal fluid within the ventricles of the brain, the dural tube, and the central nervous system surrounding the spinal and cranial nerves. The therapist uses specific techniques to help restore physiological balance of the nervous system, the brain and cranial bones. Homeostasis is restored to t
Source: case-report-1.pdf
Page: 1


Load the DPR encoders

In [ ]:
import torch
import numpy as np

from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Using device:", device)

ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained(
    "facebook/dpr-ctx_encoder-single-nq-base"
)
ctx_model = DPRContextEncoder.from_pretrained(
    "facebook/dpr-ctx_encoder-single-nq-base"
).to(device)

q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
)
q_model = DPRQuestionEncoder.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
).to(device)

ctx_model.eval()
q_model.eval()

Using device: cpu


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DPRQuestionEncoder(
  (question_encoder): DPREncoder(
    (bert_model): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=76

Generate document embeddings

In [ ]:
def encode_documents(documents, batch_size=8):
    all_embeddings = []

    texts = [doc["text"] for doc in documents]

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = ctx_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            embeddings = ctx_model(**inputs).pooler_output

            # Normalize for cosine-similarity search.
            embeddings = torch.nn.functional.normalize(
                embeddings, p=2, dim=1
            )

        all_embeddings.append(
            embeddings.cpu().numpy().astype("float32")
        )

    return np.concatenate(all_embeddings, axis=0)


doc_embeddings = encode_documents(documents)

print("Embedding shape:", doc_embeddings.shape)

Embedding shape: (78, 768)


Build the FAISS index

In [ ]:
import faiss

dimension = doc_embeddings.shape[1]

# Inner product between normalized vectors = cosine similarity.
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

print("FAISS index created.")
print("Number of indexed chunks:", index.ntotal)

FAISS index created.
Number of indexed chunks: 78


Retrieve the most relevant chunks

In [ ]:
def search(query, k=4):
    q_inputs = q_tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    q_inputs = {
        key: value.to(device)
        for key, value in q_inputs.items()
    }

    with torch.no_grad():
        q_embedding = q_model(**q_inputs).pooler_output
        q_embedding = torch.nn.functional.normalize(
            q_embedding, p=2, dim=1
        )

    q_vector = q_embedding.cpu().numpy().astype("float32")

    k = min(k, index.ntotal)
    scores, indices = index.search(q_vector, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        doc = documents[int(idx)]

        results.append({
            "text": doc["text"],
            "source": doc["source"],
            "page": doc["page"],
            "score": float(score)
        })

    return results

Test retrieval before calling the LLM

In [ ]:
query = (
    "What is lightheadedness and how is it related "
    "to rheumatoid arthritis?"
)

retrieved_docs = search(query, k=4)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved chunk {i} ---")
    print("Source:", doc["source"])
    print("Page:", doc["page"])
    print("Similarity:", round(doc["score"], 4))
    print(doc["text"][:700])


--- Retrieved chunk 1 ---
Source: case-report-2.pdf
Page: 1
Similarity: 0.6425
and muscle weakness. Patient began experiencing symptoms 4 months ago (November 2017). At that time he experienced fatigue and joint pain in the knees and hands. He was diagnosed with Rheumatoid Arthritis. He was given a short course of corticosteroids at that time that alleviated his symptoms. He was also started on methotrexate at that time. However, he felt that the medication was ineffective and stopped after 2 weeks. For the past two months, the patient has been experiencing worsening symptoms. He has been experiencing progressively worsening headaches accompanied with lightheadedness, light and sound sensitivity, nausea, and vomiting. He reports no loss of consciousness associated with

--- Retrieved chunk 2 ---
Source: case-report-1.pdf
Page: 6
Similarity: 0.6274
6 5 4 3 2 1 0 Dorsiflexion Plantarflexion Inversion Eversion Fig. 2 Standard Manual Muscle Testing pre- (blue) and post-craniosacral therap

Connect to NVIDIA NIM

In [ ]:
from openai import OpenAI
from getpass import getpass

nvidia_api_key = getpass(
    "Enter your NVIDIA NIM API key: "
)

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=nvidia_api_key
)

MODEL_NAME = "openai/gpt-oss-20b"

print("NVIDIA NIM client initialized.")

Generate the RAG answer using NVIDIA NIM

In [ ]:
def generate_rag_answer(query, retrieved_docs):
    if not retrieved_docs:
        return "No relevant document passages were retrieved."

    context_parts = []

    for i, doc in enumerate(retrieved_docs, start=1):
        context_parts.append(
            f"[Source {i}: {doc['source']}, page {doc['page']}]\n"
            f"{doc['text']}"
        )

    context = "\n\n".join(context_parts)

    system_prompt = """
You are a document-grounded question-answering assistant.

Answer the user's question using the supplied document passages.
Do not invent facts that are not supported by the passages.
If the passages do not contain enough information, say so clearly.
Cite the source filename and page number when making relevant claims.
For medical questions, provide general information only and do not
diagnose the user or replace professional medical advice.
"""

    user_prompt = f"""
Question:
{query}

Retrieved document passages:
{context}

Write a clear, concise answer based on these passages.
Include source filename and page references where appropriate.
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=500,
        temperature=0.1
    )

    return response.choices[0].message.content.strip()


answer = generate_rag_answer(query, retrieved_docs)
print("Question:", query)
print("\nRAG Answer:\n")
print(answer)

Question: What is lightheadedness and how is it related to rheumatoid arthritis?

RAG Answer:

**Lightheadedness** is a sensation of dizziness or feeling faint.  
In the case report, the patient with rheumatoid arthritis (RA) described experiencing lightheadedness **alongside** severe headaches, nausea, vomiting, and worsening joint pain. The lightheadedness appeared during periods of increased disease activity and was noted as part of the patient’s overall symptom flare (Source 1: *case‑report‑2.pdf*, page 1; Source 4: *case‑report‑2.pdf*, page 1).  

Thus, in this patient, lightheadedness is a symptom that accompanies RA flare‑ups, likely reflecting the systemic inflammatory burden or side‑effects of treatment rather than a separate condition.
